# Pingüinos — Sesión 1: nulos y calidad de datos**Fase 1 · Pandas** — Dataset: Palmer Penguins (344 filas, 7 columnas)Antes de analizar nada, compruebo de qué datos dispongo **realmente**:dónde faltan, si faltan por alguna razón, y si descartarlos cambia las conclusiones.

In [ ]:
import pandas as pdimport seaborn as snsimport sys; sys.path.append("../src")from utils import media_pordf = sns.load_dataset("penguins")

## 1. Valido mi función contra PandasComparo `media_por`, que escribí a mano en la Fase 0, con el `groupby` de Pandas.Si dan lo mismo, confirmo que entiendo qué hace Pandas por debajo.

In [ ]:
# Las dos líneas agrupan por la MISMA columna: solo así la comparación tiene sentido.print(media_por(df.to_dict("records"), "species", "body_mass_g"))print(df.groupby("species")["body_mass_g"].mean().round(2))# Coinciden. Detalle que conviene no olvidar: mi función ignora los nan porque yo# lo programé con un continue. Pandas toma esa misma decisión por defecto,# pero sin avisar de que la está tomando.

## 2. ¿Dónde faltan datos?

In [ ]:
df.info()

In [ ]:
# Forma directa de ver los huecos por columna, sin leer el info() a ojo.print(df.isna().sum())# 4 columnas de medidas con 2 huecos cada una: son 2 pingüinos sin medir en absoluto.# 'sex' es el caso distinto: 11 huecos, y 9 de ellos son pingüinos con TODAS# las medidas tomadas. No son el mismo tipo de hueco y no merecen el mismo trato.

## 3. `len()` cuenta filas, `count()` cuenta datosLas dos líneas de abajo parecen calcular lo mismo y dan números distintos.

In [ ]:
print(df["body_mass_g"].mean())print(df["body_mass_g"].sum() / len(df))# La trampa: sum() TAMBIÉN ignora los nulos, igual que mean(). El numerador es# idéntico en las dos líneas. Lo único que cambia es el denominador:#   mean()  divide entre 342  -> los valores que existen de verdad#   len(df) son 344           -> las filas de la tabla, tengan dato o no## El error es del 0,6 %. No sale nan, no salta ningún aviso: sale un número# perfectamente creíble. El denominador correcto sería df["body_mass_g"].count()

## 4. Agrupar descarta filas en silencio

In [ ]:
# Por defecto groupby TIRA las filas cuyo valor de agrupación es nulo: los 11# pingüinos sin sexo desaparecen y la tabla suma 333, no 344. dropna=False los conserva.print(df.groupby(["species", "sex"], dropna=False)["body_mass_g"].agg(["mean", "count"]).round(2))# Pido mean y count juntos a propósito: una media sin saber sobre cuántos datos# está calculada es medio dato. Aquí se ve que los grupos NaN tienen 5 y 4# pingüinos, así que esas dos medias son anécdotas, no resultados.## Nota: dropna=False afecta a los nulos de las columnas por las que AGRUPO.# Los nulos de body_mass_g los sigue ignorando mean(), eso no cambia.# Y 'count' no cuenta registros: cuenta valores no nulos.

## 5. ¿Descartar a los que no tienen sexo sesga el análisis?Si esos 9 pingüinos se parecieran poco al resto, quitarlos deformaría los resultados.Lo compruebo **antes** de decidir, no después.

In [ ]:
sin_sexo = df[df["sex"].isna()]con_sexo = df[df["sex"].notna()]print(sin_sexo["body_mass_g"].agg(["mean", "count"]))print(con_sexo["body_mass_g"].agg(["mean", "count"]))# 4005 g frente a 4207 g: 201 g de diferencia, un 4,8 %. ¿Eso es mucho o poco?

### ¿201 g es una diferencia grande?Dos formas de contestar sin estadística formal: compararla con la variaciónnatural del dataset, y ver cuánto baila una media calculada con solo 9 datos.

In [ ]:
# Desviación típica: cuánto se separan los pingüinos INDIVIDUALES del peso medio.# Ojo con la definición: no mide cuánto "se desvía la media" (la media es un punto# fijo), mide la dispersión de los datos alrededor de ella.print(df["body_mass_g"].std())   # ~802 g# Cinco medias de 9 pingüinos elegidos AL AZAR, sin ningún sesgo de por medio:for i in range(5):    muestra = df["body_mass_g"].dropna().sample(9)    print(round(muestra.mean(), 2))# Salen valores entre 3722 y 4300 solo por azar. Mi 4005 cae justo en medio.# Los 201 g son además una cuarta parte de los 802 g de variación normal.# Conclusión: la diferencia no significa nada, puedo descartar los 9 sin problema.

## 6. ¿Los huecos siguen algún patrón?Si los datos faltasen por una razón sistemática, descartarlos podría sesgar elresultado sin que nada me avisara. Miro dónde se concentran.

In [ ]:
# Dónde están los 11 sin sexo, cruzando especie e islaprint(df[df["sex"].isna()].groupby(["species", "island"]).size().reset_index(name="sin_sexo"))# Cuántos pingüinos hay en cada isla, para poder comparar en porcentajeprint(df.groupby("island").size())

In [ ]:
# Porcentaje de pingüinos SIN SEXO ANOTADO dentro de cada isla.# El denominador es la isla, no el dataset entero: solo así son comparables# islas de tamaños distintos (5 huecos sobre 52 no es lo mismo que 5 sobre 168).porcentaje = df["sex"].isna().groupby(df["island"]).mean() * 100print(porcentaje.round(1))# Torgersen 9,6 % frente a Dream 0,8 %: doce veces más. Eso no es azar, hubo# algún problema de campo en esa isla.# Pero NO me afecta: Torgersen solo tiene Adelie, y hay Adelie en las otras dos# islas. Descartarlos adelgaza la muestra (151 -> 146) sin eliminar ningún grupo.

## 7. El truco de los booleanosEn Python `True` vale 1 y `False` vale 0. Sobre una columna de condiciones:`.sum()` cuenta **cuántos** cumplen, `.mean()` da la **proporción** que cumple.

In [ ]:
print((df["island"] == "Torgersen").sum())    # 52  -> cuántos pingüinos cumplenprint((df["island"] == "Torgersen").mean())   # 0.151 -> PROPORCIÓN, no porcentaje# Para leerlo como porcentaje hay que multiplicar por 100 (15,1 % del dataset).# El denominador de .mean() siempre son TODAS las filas del DataFrame.## El patrón funciona con cualquier condición, no solo con nulos:#   (df["body_mass_g"] > 5000).mean() * 100   -> % de pingüinos de más de 5 kg

## Conclusiones1. Faltan datos en 5 columnas. Son dos situaciones distintas: **2 pingüinos sin ninguna   medida** (filas inservibles) y **9 con todas las medidas pero sin sexo anotado**   (información aprovechable).2. Por tanto **descarto por columna, no por fila**: `dropna(subset=[...])` con las   columnas que necesite cada pregunta. Un `dropna()` a secas tiraría 9 pingüinos   buenos incluso en análisis donde el sexo no pinta nada.3. Los que no tienen sexo pesan 201 g menos, pero la diferencia **cabe dentro del   azar** de una muestra de 9. No sesga.4. Los huecos **sí tienen patrón**: se concentran en Torgersen. No afecta a estos   análisis porque esa isla solo tiene Adelie, presentes también en las otras dos.5. Aprendizaje transversal: Pandas toma decisiones por mí (ignorar nulos al sumar,   descartar filas al agrupar) **sin avisar**. Los resultados salen creíbles igual.   Hay que saber qué está decidiendo en mi nombre.